In [1]:
import sys
import os
import webbrowser
from tkinter import Tk, filedialog

import folium
from folium.plugins import TagFilterButton, FloatImage, OverlappingMarkerSpiderfier

import fireducks.pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from datetime import date

In [2]:
def get_working_dir():
    """When frozen (PyInstaller) use exe dir; else current dir."""
    if getattr(sys, 'frozen', False):
        return os.path.dirname(sys.executable)
    return os.path.abspath(".")

In [3]:
def choose_file_or_default(default_name="Filiera batterie Motus-E Italia.xlsx"):
    # Small GUI for non-technical user to pick file; fallback to default in same folder
    root = Tk()
    root.withdraw()
    path = filedialog.askopenfilename(title="Choose your EXCEL file",
                                      filetypes=[("Excel files", "*.xlsx *.xls")])
    return path if path else os.path.join(get_working_dir(), default_name)

In [4]:
def choose_json(default_name="it.json"):
    # Small GUI for non-technical user to pick file; fallback to default in same folder
    root = Tk()
    root.withdraw()
    path = filedialog.askopenfilename(title="Choose your JSON file",
                                      filetypes=[("json files", "*.json")])
    
    if not os.path.exists(path):
        print("JSON not found: it.json")
        return    
    
    return path if path else os.path.join(get_working_dir(), default_name)

In [5]:
def process_and_render(excel_path):
    
    df = pd.read_excel(excel_path, engine="openpyxl")
    df = df.replace([None,np.nan], value=" ")
    df[["Lat","Lon"]] = df["Lat,Lon"].str.split(",", expand=True)
    df["Lat"] = pd.to_numeric(df["Lat"], errors="coerce")
    df["Lon"] = pd.to_numeric(df["Lon"], errors="coerce")

    m = folium.Map((41.89, 12.48), zoom_start=6, 
               tiles="cartodb positron",
               max_bounds=True,
               scrollWheelZoom=False,
               min_lat=35.83205927016135,
               max_lat=48.155920623358526,
               min_lon=-4.812802161016015,
               max_lon=26.461693176182866)
    geo_json_data = choose_json()
    folium.GeoJson(geo_json_data,
                  style_function=lambda feature: {
                      "fillColor": "#00c9a9",
                      "color": "black",
                      "weight": 1,
                      "dashArray": "5, 5",
                  },
                  highlight_function=lambda feature: {"fillColor": "#edfdf9"}).add_to(m)
    
    categories = list(pd.read_excel(excel_path, sheet_name="Dati")["Filiera"])
    motus_logo = "https://www.motus-e.org/wp-content/uploads/2022/01/logo-dark.png"
    #FloatImage(motus_logo, bottom=5, left=75).add_to(m)
    
    
    for azienda, lat, lon, tag1, tag2, tag3, attivo, link in zip(df["Azienda"], df["Lat"], df["Lon"], df["Filiera 1"], df["Filiera 2"], df["Filiera 3"], df["In produzione"], df["Sito Web"]):
    
        text = f"""<h1><a href="{link}" target="_blank"><span style="font-family:'Montserrat';color:#006fb8">{azienda}</span></a></h1>
            <br><span style="font-family:'Montserrat'">
            -{tag1}<br>"""
        if tag2 != " ":
            text += f"-{tag2}<br>"
        if tag3 != " ":
            text += f"-{tag3}<br>"
    
        if attivo == "Sì":
            color = "green"
        else:
            color = "lightgray"
            text += f"<br><i>>>In costruzione<<</i>"
        text += "</span>"
    
        
        popup = folium.Popup(text, max_width=200)
        
        folium.Marker(location=[lat,lon], popup=popup, tags=[tag1, tag2, tag3], 
                      tooltip=azienda, icon=folium.Icon(icon="flash", color=color)).add_to(m)
    
    oms = OverlappingMarkerSpiderfier(
        keep_spiderfied=False,  # Markers remain spiderfied after clicking
        nearby_distance=20,  # Distance for clustering markers in pixel
        circle_spiral_switchover=10,  # Threshold for switching between circle and spiral
        leg_weight=2.0  # Line thickness for spider legs
        )
    oms.add_to(m)

    TagFilterButton(categories, clear_text="Deseleziona tutto").add_to(m)
    folium.FitOverlays().add_to(m)

    today = date.today()
    today_iso = date.isoformat(today)

    m.save(f"Motus-E_Filiera_batterie_{today_iso}.html")

In [6]:
def main():
    excel = choose_file_or_default("Filiera batterie Motus-E Italia.xlsx")
    if not os.path.exists(excel):
        print("Excel not found:", excel)
        return
    process_and_render(excel)

if __name__ == "__main__":
    main()

/home/pogechi/anaconda3/envs/teraenv/lib/python3.12/site-packages/openpyxl/worksheet/_read_only.py:85: UserWarning: Data Validation extension is not supported and will be removed
  for idx, row in parser.parse():
